# COVID-19 in England, winter 2020–21: rate maps

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leehart3000/epiplot/blob/main/docs/case-studies/covid-england-2020-21.ipynb)

In the winter of 2020–21, England had its largest wave of COVID-19 cases up to
that point. A new variant, later named Alpha, had been first detected in Kent in
the autumn. Over the following weeks it spread from the south-east across the
whole country.

This case study maps that wave by local authority, using real case counts and
populations. A **rate map** shades each area by its rate of cases per 100,000
people, so places with very different populations can be compared fairly.

## Set up

If you are running this notebook in Google Colab, the next cell installs
**epiplot** with its map libraries first. Elsewhere it does nothing.

Colab will warn that the notebook was not authored by Google. It shows this for any
notebook opened from GitHub. This one only installs **epiplot** and reads public data
files: it does not ask for access to your Google account or data.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install --quiet "epiplot[maps]"

In [ ]:
from pathlib import Path

import geopandas
import matplotlib.pyplot as plt
import pandas as pd

import epiplot

## Load the data

There are two files, prepared from public UKHSA and ONS data (see the notes at
the end):

- **Cases and populations:** one row per local authority per period, with the
  number of new cases (by date the test sample was taken) and the mid-2020
  population.
- **Boundaries:** the shape of each local authority, as at December 2020.

In [ ]:
FOLDER = Path("data")
if not FOLDER.exists():  # for example in Colab: read the copies on GitHub instead
    FOLDER = (
        "https://raw.githubusercontent.com/leehart3000/epiplot/main/"
        "docs/case-studies/data"
    )

data = pd.read_csv(f"{FOLDER}/covid_england_2020_21.csv")
regions = geopandas.read_file(f"{FOLDER}/england_lad_2020.geojson")
data.head()

## Equal periods, so rates compare fairly

A rate depends on how long you count for: twice as many days gives roughly twice
as many cases. Calendar months would be a trap here, because February 2021 had
28 days and January 31, so February would look about 10% lower for no real
reason.

So the data use **four equal 28-day periods**, each starting on a Monday. Making
the period column *categorical*, in date order, keeps the maps in time order
rather than alphabetical order.

In [ ]:
periods = data.drop_duplicates("period").sort_values("period_start")["period"]
data["period"] = pd.Categorical(data["period"], categories=periods)
list(data["period"].cat.categories)

## Use a map projection that keeps areas fair

The boundary file stores positions as longitude and latitude. Drawn directly,
that stretches England sideways and distorts the size of each area, which
matters on a map where the eye reads large shaded areas as "a lot". **epiplot**
warns if you forget. Converting to the British National Grid (EPSG:27700), the
standard projection for Great Britain, fixes it.

In [ ]:
regions = regions.to_crs(27700)

## One period: the peak

Before looking at the whole winter, here is a single map of the busiest period.

In [ ]:
peak = data[data["period"] == "28 Dec – 24 Jan"]
ax = epiplot.rate_map(
    regions,
    peak,
    "area_code",
    cases_col="cases",
    population_col="population",
)
ax.set_title("COVID-19 cases per 100,000 people, 28 Dec 2020 – 24 Jan 2021")
plt.show()

Small areas, such as the London boroughs, are hard to see on a map of the whole
country. A table of the highest rates fills in the detail, with exact 95%
confidence intervals:

In [ ]:
rates = epiplot.calculate_rates(peak, "area_code", "cases", "population")
rates = rates.merge(peak[["area_code", "area_name"]], on="area_code")
top = rates.sort_values("rate", ascending=False).head(10)
top[["area_name", "cases", "population", "rate", "rate_lower", "rate_upper"]].round(0)

## The whole winter: a timeline

Four small maps, one per period, all sharing **one colour scale**: the same
shade means the same rate in every map, so the maps can be compared directly.

In [ ]:
figure = epiplot.rate_map_timeline(
    regions,
    data,
    "area_code",
    "period",
    cases_col="cases",
    population_col="population",
    ncols=4,
)
figure.suptitle(
    "COVID-19 cases per 100,000 people in each 28-day period, England, winter 2020–21"
)
plt.show()

## What the maps show

- **2–29 November:** during England's second national lockdown, the highest
  rates were in the north, in parts of Lancashire, Yorkshire and the Humber.
- **30 November – 27 December:** the north eased, but rates climbed sharply in
  Kent, Essex and east London, where the Alpha variant was spreading fastest.
  London and much of the south-east moved into the new, strictest "Tier 4" on
  20 December.
- **28 December – 24 January:** the peak. High rates spread across almost the
  whole country. A third national lockdown began on 6 January.
- **25 January – 21 February:** rates fell everywhere.

Maps like these show *where* and *when*, not *why*. Rates also depend on how
much testing was done, which varied between places and over time.

## Good practice, built in

- **Rates, not counts.** Raw case counts mostly show where people live. Rates
  per 100,000 people allow fair comparison between areas of very different
  size. The colour bar says they are *crude* rates, not adjusted for age.
- **One colour scale for all four maps,** starting at zero, in a single colour
  that gets darker as the rate rises.
- **Small numbers would be hidden.** Areas with fewer than 5 cases in a period
  would be shaded as "hidden", because such rates are unreliable and could
  identify individuals. This winter, every area had more than that.
- **Missing data would be shown differently from zero,** so a gap in reporting
  is never mistaken for an absence of disease.

## Try it yourself

- Use `bands=5` to show the rates in five colour bands instead of a smooth
  scale.
- Draw `rate_map` for another period, such as `"30 Nov – 27 Dec"`.
- Sort the `rates` table the other way, to find the areas with the lowest rates.

## Data sources and licences

- **Cases:** UKHSA COVID-19 archive data, new cases by specimen date for each
  lower-tier local authority.
  <https://ukhsa-dashboard.data.gov.uk/covid-19-archive-data-download>.
  Contains public sector information licensed under the Open Government
  Licence v3.0.
- **Boundaries:** ONS, Local Authority Districts (December 2020) Boundaries UK
  BUC. Source: Office for National Statistics licensed under the Open
  Government Licence v3.0. Contains OS data © Crown copyright and database
  right 2020.
- **Population:** ONS mid-2020 population estimates. Source: Office for
  National Statistics licensed under the Open Government Licence v3.0.

To match the three sources, the Isles of Scilly are joined with Cornwall and the
City of London with Hackney (UKHSA reported their cases together), and the four
former Buckinghamshire districts are combined. The script that prepared the
data, with notes on every change, is in the
[data folder on GitHub](https://github.com/leehart3000/epiplot/tree/main/docs/case-studies/data).